# Act 1 — Decoupling: SQS and SNS

How do components of a distributed system pass work to each other without being tightly coupled? The naive approach is a direct A-P-I call from producer to consumer, but that ties their fates together — if the consumer is slow, the producer slows. If the consumer is down, the producer fails. If you add a second consumer, the producer has to know about it.

The two foundational decoupling patterns are a queue — where one consumer processes each message at its own pace — and pub/sub — where every subscriber gets a copy of every message. **SQS** is the queue. **SNS** is the pub/sub. The two combine into the most reused pattern in AWS messaging, and EventBridge in Act 4 is a richer cousin of SNS. This act covers all of the messaging side.

## Five services, one question

Five services in this notebook, all answering different versions of the same question: how do components of a distributed system pass work to each other without being tightly coupled? SQS is a queue — producers drop messages in, consumers pull them out at their own pace. SNS is pub/sub — one publish, every subscriber gets a copy. Kinesis and MSK are real-time streams for high-volume, multi-consumer event data. Step Functions is workflow orchestration — a managed state machine that sequences tasks, branches, retries. EventBridge is an event bus that routes AWS-service events, SaaS events, and custom application events to targets by rules.

The pattern across all of them is the same: get producers and consumers off each other's hot path, let each side scale independently, survive the failure of either, and add observability and retries for free. The differences are in shape — one-to-one queue vs one-to-many fan-out vs persistent stream vs orchestrated workflow vs rule-routed bus — and the skill is matching the shape to the problem.

## SQS — The Queue

SQS is a fully managed message queue. A producer calls `SendMessage`; the message sits in the queue. A consumer calls `ReceiveMessage`, gets back up to ten messages, does its work, then calls `DeleteMessage` to remove each one from the queue. If the consumer never deletes — because it crashed, timed out, or threw — the message reappears in the queue after the **visibility timeout** and another consumer picks it up.

That reappearance is the heart of SQS's at-least-once guarantee. While a message is being processed, it's hidden from other consumers, so two workers don't both run the same job. If processing succeeds, the consumer deletes the message and it's gone. If processing fails for any reason short of an explicit delete, the visibility timer expires and the message is delivered again. Consumers can extend the timeout with `ChangeMessageVisibility` if processing turns out to take longer than expected.

| Setting | Default | Range |
|---|---|---|
| Message size | — | up to 256 KB (use S3 + pointer for more) |
| Message retention | 4 days | 1 minute – 14 days |
| Visibility timeout | 30 s | 0 – 12 hours |
| Delivery delay | 0 | 0 – 15 minutes |
| Receive wait time | 0 s | 0 – 20 s (long polling) |
| Batch size per receive | — | 1 – 10 messages |

The single most important rule is **set visibility timeout longer than the maximum processing time**. If processing routinely takes 90 seconds but the timeout is 30, every message gets delivered to a second consumer while the first is still working — duplicate work, broken invariants, downstream chaos. Pick a timeout that comfortably exceeds the 99th-percentile processing time, and use `ChangeMessageVisibility` for the unusual long ones.

## Standard vs FIFO Queues

SQS comes in two flavours. The choice is almost always driven by whether you need ordering.

| | Standard | FIFO |
|---|---|---|
| Ordering | best-effort | **strict per message group** |
| Delivery | at-least-once (occasional duplicates) | **exactly-once** processing |
| Throughput | effectively unlimited | 300 msg/s (3,000 with batching, higher with high-throughput mode) |
| Name suffix | any | must end in `.fifo` |

**Standard** is the default — massive throughput, occasional duplicates, no ordering guarantee. Most asynchronous-work workloads fit here: image processing, email sending, log shipping. The application has to be idempotent (handle the same message being processed twice without corrupting state), which is good design anyway.

**FIFO** trades throughput for ordering and exactly-once processing. Two concepts come with it.

- **Message Group ID.** Messages sharing a group ID are delivered in strict order; different group IDs are processed independently in parallel. The group ID is how FIFO scales — pick `customerId` and customers process in parallel while each customer's messages stay ordered.
- **Message Deduplication ID.** SQS deduplicates messages with the same ID within a five-minute window — so retried `SendMessage` calls from a flaky network don't create duplicates. **Content-based deduplication** auto-computes the dedup ID from a SHA-256 hash of the message body, which is the right default for most idempotent producers.

The decision rule: order matters or duplicates would corrupt state → FIFO. Otherwise → Standard.

## DLQs, Polling, Delays, Large Messages

A few SQS features that come up constantly.

**Dead-Letter Queue (DLQ).** A separate queue where SQS sends messages that fail processing too many times. Configure `maxReceiveCount` on the main queue (commonly 3 or 5); after that many failed deliveries (i.e. the message was received but never deleted, that many times), SQS moves the message to the DLQ. The DLQ has to be the same type as the source — FIFO source needs FIFO DLQ. Two reasons this matters. First, **poison messages** (malformed payloads that break the consumer every time) don't block the queue — they're shunted aside after a few attempts so the rest of the queue keeps moving. Second, the DLQ is the audit log of what failed: inspect it, fix the bug, and use **DLQ redrive** to replay the messages back into the source queue.

**Long polling.** Set `WaitTimeSeconds` from 1 to 20 on `ReceiveMessage`. Instead of returning immediately when the queue is empty (short polling, the default), SQS waits up to that duration for a message to arrive. Reduces empty responses, lowers API costs, reduces CPU spin on the consumer. There's no reason not to use long polling for any real consumer — always set it.

**Delay queues.** Set a delivery delay (0–15 minutes) on the queue or per-message. New messages are invisible to consumers until the delay expires. Useful for "do this in ten minutes" patterns without a separate scheduler.

**Large messages.** SQS payloads are capped at 256 KB. For larger items, the convention is to put the payload in S3 and send a pointer (the S3 key) through SQS. The **SQS Extended Client Library** (Java, Python) does this transparently — the producer hands it a 5 MB blob and it stores the body in S3 and the pointer in SQS automatically.

Security wraps the rest: encryption at rest via KMS (server-side encryption), TLS in transit, IAM policies on the API and **queue policies** (resource-based) for cross-account access and for letting services like SNS or S3 publish into the queue. VPC interface endpoints let you call SQS without traversing the public internet.

## SNS — Pub/Sub

SNS is a fully managed pub/sub service. The model is one-to-many: a publisher sends a message to a **topic** and SNS delivers a copy to every **subscriber** simultaneously. Unlike SQS, there's no queue — once delivered, the message is gone from SNS's perspective. Persistence is the subscriber's job.

Subscribers can be:

- **SQS queues** — by far the most common; turns SNS into a persistent fan-out
- **Lambda functions** — event-driven processing
- **HTTP / HTTPS endpoints** — webhooks
- **Email / Email-JSON, SMS, Mobile Push** (APNS, FCM)
- **Kinesis Data Firehose** — for delivery into S3 / Redshift / OpenSearch

A single topic can have up to **12.5 million subscriptions**, and an account can hold up to 100,000 topics. Message size is capped at 256 KB, same as SQS.

**Filter policies** let each subscriber declare what it cares about. By default, every subscriber gets every message; with a filter policy attached, the subscriber only receives messages whose attributes (or body, with body-based filtering) match. Conditions support exact match, prefix, numeric ranges, existence, anything-but. The architectural point: a single `order-events` topic can serve a dozen microservices, each subscribing with its own filter, instead of you provisioning a separate topic per event type.

**SNS FIFO topics** are the FIFO equivalent. Strict ordering per message group, exactly-once delivery — but with a hard constraint: SNS FIFO can only deliver to **SQS FIFO queues**. No Lambda, no HTTP, no email. If you need ordered fan-out, the stack is SNS FIFO → SQS FIFO.

SNS Standard tops out at very high throughput; SNS FIFO is capped at 300 messages per second per topic (3,000 with batching), the same as SQS FIFO.

## The SNS + SQS Fan-Out Pattern

The most reused pattern in AWS messaging. One event needs to reach several downstream systems, each at its own pace, each with its own retry behaviour. Publishing direct to multiple SQS queues from the producer would couple the producer to the fan-out topology — add a consumer, change the producer; one SQS write fails, partial delivery.

Instead:

- Producer publishes once to an SNS topic
- SNS delivers atomically to each subscribed SQS queue
- Each SQS queue feeds an independent consumer (Lambda, ECS worker, EC2 fleet)
- Each queue has its own DLQ, visibility timeout, and scaling

The producer doesn't know who's downstream. Adding a new consumer is one new SQS queue and one new SNS subscription — no producer change. Each consumer survives its peers' failures, because the queues are independent.

The canonical use cases are everywhere: an order placement that has to email the customer, charge the card, reserve inventory, and notify shipping; an S3 upload that should trigger image processing, virus scanning, and analytics indexing; a user signup that has to provision a tenant, send a welcome email, and write to a CRM. **S3 event notifications** in particular only support a single direct destination per event type, so when several consumers need to react to S3 events the standard pattern is S3 → SNS → multiple SQS queues.

The filter policy layer makes this more precise: subscribers attach a filter so a single topic carrying many event types only delivers the relevant ones to each queue. The result is a small number of topics serving many consumers — closer to a routing layer than to a fan-out cable.

## SQS vs SNS vs EventBridge — the messaging triangle

Three services that all decouple producers from consumers and look similar in shape. Picking the right one is a frequent decision.

| | SQS | SNS | EventBridge |
|---|---|---|---|
| Model | queue (pull) | pub/sub (push) | event bus with rules |
| Consumers | one per message | many simultaneously | many via rules |
| Persistence | yes — up to 14 days | no | optional archive |
| Filtering | none (DLQ only) | per-subscription attribute filters | rich pattern matching |
| Sources | applications | applications | AWS services + SaaS + custom |
| Throughput | very high | very high | high, with rule overhead |
| Cost | cheapest | cheap | a little more |

The practical decision tree:

- **Decouple producer from a single consumer, buffer traffic** → SQS
- **One event, many downstream subscribers, simple delivery** → SNS (often paired with SQS for the fan-out durability)
- **Routing AWS service events or SaaS events with complex matching rules, or many-to-many across accounts/regions** → EventBridge

EventBridge is covered in detail later in this notebook — it's the same idea as SNS with a richer matching layer and a broader source surface.

In [ ]:
import boto3, json

sqs = boto3.client("sqs", region_name="us-east-1")
sns = boto3.client("sns", region_name="us-east-1")

# DLQ first, then main queue with a redrive policy pointing at it
dlq = sqs.create_queue(QueueName="orders-dlq")
dlq_arn = sqs.get_queue_attributes(QueueUrl=dlq["QueueUrl"], AttributeNames=["QueueArn"])["Attributes"]["QueueArn"]

main = sqs.create_queue(
    QueueName="orders",
    Attributes={
        "VisibilityTimeout":            "60",       # > 99th-pct processing time
        "ReceiveMessageWaitTimeSeconds": "20",       # long polling — always on
        "RedrivePolicy": json.dumps({"deadLetterTargetArn": dlq_arn, "maxReceiveCount": 3}),
        "SqsManagedSseEnabled":         "true",
    },
)
queue_url = main["QueueUrl"]

# Send + receive + delete
sqs.send_message(QueueUrl=queue_url, MessageBody=json.dumps({"orderId": "ORD-001"}),
                 MessageAttributes={"eventType": {"DataType": "String", "StringValue": "ORDER_PLACED"}})

resp = sqs.receive_message(QueueUrl=queue_url, MaxNumberOfMessages=10, WaitTimeSeconds=20,
                           MessageAttributeNames=["All"])
for msg in resp.get("Messages", []):
    process(json.loads(msg["Body"]))                # idempotent worker
    sqs.delete_message(QueueUrl=queue_url, ReceiptHandle=msg["ReceiptHandle"])

# SNS topic with the SQS queue subscribed — fan-out with an attribute filter
topic = sns.create_topic(Name="order-events")["TopicArn"]
queue_arn = sqs.get_queue_attributes(QueueUrl=queue_url, AttributeNames=["QueueArn"])["Attributes"]["QueueArn"]
sns.subscribe(
    TopicArn=topic, Protocol="sqs", Endpoint=queue_arn,
    Attributes={"FilterPolicy": json.dumps({"eventType": ["ORDER_PLACED", "ORDER_CANCELLED"]})},
)

# FIFO queue — strict per-group ordering plus content-based deduplication
sqs.create_queue(
    QueueName="payments.fifo",
    Attributes={"FifoQueue": "true", "ContentBasedDeduplication": "true", "VisibilityTimeout": "60"},
)

# Act 2 — Streaming: Kinesis and MSK

SQS is great for asynchronous work — one consumer per message, delete when done. But what if you have telemetry, clickstreams, IoT signals, or change-capture data that needs to feed *multiple* downstream consumers, each at their own pace, with the ability to rewind and replay history? That is a stream, not a queue.

**Kinesis Data Streams** is the AWS-native streaming service. **Firehose** is the managed delivery sibling that pipes streaming data into S3, Redshift, or OpenSearch without any consumer code. **Managed Service for Apache Flink** runs windowed analytics over those streams. And **MSK** is managed Apache Kafka for teams that already speak Kafka. This act walks each one, then the canonical Kinesis-versus-MSK decision.

## Kinesis Data Streams

Kinesis Data Streams is a real-time streaming service. Producers push records into a stream; multiple consumers read them at their own pace, with records retained in the stream for up to a year.

The difference from SQS matters. SQS is a work queue — each message is processed by one consumer and then deleted. Kinesis is a stream — records stay in the stream after consumption, multiple consumers can read the same data independently, and consumers can rewind and replay. SQS is the right shape for asynchronous jobs; Kinesis is the right shape for telemetry, clickstreams, IoT, change-capture, anything where the data has multiple downstream consumers or has to be reprocessed.

### Shards

A stream is divided into **shards** — the unit of capacity. Each shard supports:

- **1 MB/s or 1,000 records/s on write**
- **2 MB/s on read** (shared across all standard consumers)

Scale by adding shards (splitting) or removing them (merging). Every record carries a **partition key** that Kinesis hashes to choose a shard — high-cardinality partition keys distribute load evenly across shards, low-cardinality keys create hot shards that throttle while the rest sit idle (the same pattern as DynamoDB).

### Retention

Default retention is **24 hours**, extendable to **7 days** at small extra cost, or up to **365 days** at higher cost. Within the retention window any consumer can replay from any point — `TRIM_HORIZON` for the oldest available, `LATEST` for new records only, or `AT_TIMESTAMP` / `AT_SEQUENCE_NUMBER` for precise restart.

### Capacity modes

- **Provisioned** — you set shard count and manage scaling. Cheaper for predictable throughput.
- **On-demand** — Kinesis auto-scales capacity, you pay per GB ingested and retrieved. Right for unpredictable or new workloads.

## Consumers — Standard vs Enhanced Fan-Out

Kinesis has two consumer models, with very different throughput characteristics.

**Standard consumers** poll the stream with `GetRecords`. The shard's 2 MB/s read budget is **shared across all standard consumers** — five consumers reading the same shard each get 400 KB/s. Latency is around 200 ms, and you're limited to five `GetRecords` calls per shard per second across all standard consumers.

**Enhanced Fan-Out (EFO)** flips the model: each EFO consumer **subscribes** to a shard and gets its own dedicated 2 MB/s — push delivery over HTTP/2, latency around 70 ms. Five EFO consumers on the same shard get five times 2 MB/s, not 2 MB/s split five ways. EFO costs more per consumer but is the right pick whenever you have several consumers that all need full throughput.

The rule: one or two consumers and cost matters → standard. Many consumers or low-latency push delivery → EFO.

### Consumer state

Consumers track their position in the stream with **sequence number checkpoints**, typically stored in DynamoDB by the **Kinesis Client Library (KCL)**. On restart, the consumer picks up from its last checkpoint. The KCL also handles shard leases — distributing shards across consumer instances, rebalancing when instances join or leave the group, and handling resharding (splits and merges) without losing the consumer's position.

The key CloudWatch metric to watch is **`IteratorAgeMilliseconds`** — how far behind the latest record a consumer is. A rising iterator age means the consumer can't keep up: scale the consumer fleet, scale the shard count, or switch the consumer to EFO.

## Kinesis Data Firehose

Firehose is the managed delivery sibling of Data Streams. Where Data Streams is a stream you build consumers against, Firehose is a one-way pipe — point it at a destination, hand it data, and Firehose delivers, buffering and batching along the way. No shards to manage, no consumers to write, no checkpointing. The trade-off is that Firehose stores nothing — it's a delivery pipe, not a stream. Records that fail delivery can be backed up to S3, but you can't replay history.

### Destinations

- **Amazon S3** — the most common; pairs with Athena or Glue downstream
- **Amazon Redshift** — Firehose writes to S3 first, then issues a `COPY` to load the cluster
- **Amazon OpenSearch Service** — real-time log and metrics dashboards
- **HTTP endpoints** — custom destinations
- **Third-party** — Splunk, Datadog, New Relic, Dynatrace, MongoDB, and others

### Buffering and transforms

Firehose batches records before delivering — by **buffer size** (1–128 MB depending on destination) or **buffer interval** (60–900 seconds), whichever is reached first. That's why Firehose is described as **near** real-time: the minimum end-to-end latency is around a minute because that's the smallest buffer interval. For sub-minute latency you need Data Streams, not Firehose.

Firehose can also **transform records inline** with a Lambda function before delivery — common patterns are JSON to Parquet conversion, field filtering or masking, enrichment from a lookup table, or schema fixes. JSON-to-Parquet conversion is a built-in option that uses the Glue Data Catalog to pick up the schema.

### Picking between them

| | Data Streams | Firehose |
|---|---|---|
| Latency | real-time (ms) | near real-time (≥60 s) |
| Consumers | your code — Lambda, KCL, EFO | managed delivery to fixed destinations |
| Retention | 1–365 days | none — pipe only |
| Scaling | manual shards or on-demand | fully automatic |
| Use case | custom real-time processing, replay, fan-out | dump streaming data into S3 / Redshift / OpenSearch |

They often combine: producers write into Data Streams for the real-time consumers, and a Firehose consumer on the same stream lands a copy in S3 for the data lake.

## Data Analytics on Streams, and MSK

Two more pieces of the streaming picture.

### Managed Service for Apache Flink (formerly Kinesis Data Analytics)

Real-time analytics over streams using **Apache Flink** (Java, Scala, Python) or **SQL**. Read from Kinesis or MSK, run windowed aggregations (tumbling, sliding, session windows), join multiple streams, do anomaly detection, then write the results back to a stream or to Firehose. Fully managed — no Flink cluster to operate.

Use cases: rolling per-minute metrics for a real-time dashboard, anomaly flags when a sensor exceeds a threshold for the last N seconds, on-stream ETL that filters and enriches before landing in S3.

### Amazon MSK — Managed Streaming for Apache Kafka

MSK is fully managed Apache Kafka. Topics, partitions, brokers, consumer groups — the standard Kafka model, with AWS handling broker provisioning, patching, and multi-AZ deployment. Use existing Kafka producers, consumers, and connectors without code changes. Two flavours:

- **MSK provisioned** — you pick the broker instance type and count
- **MSK Serverless** — pay per usage, no broker management

**MSK Connect** runs managed Kafka Connect connectors to ferry data between Kafka and AWS services (S3, DynamoDB, RDS, OpenSearch) without standing up your own Connect cluster.

### Kinesis Data Streams vs MSK

| | Kinesis Data Streams | MSK |
|---|---|---|
| Protocol | AWS proprietary | Apache Kafka |
| Max record size | 1 MB | configurable, default 1 MB |
| Retention | 1–365 days | configurable, default 7 days |
| Scaling | shard split / merge or on-demand | add brokers + partitions, or Serverless |
| Ecosystem | AWS SDK, KCL | full Kafka ecosystem — Connect, Streams, ksqlDB |
| Right when... | building net-new on AWS | already on Kafka, or need Kafka-specific features |

The practical split: **Kafka in the room → MSK**. Net-new AWS-native pipeline → Kinesis.

In [ ]:
import boto3, json

kinesis  = boto3.client("kinesis",  region_name="us-east-1")
firehose = boto3.client("firehose", region_name="us-east-1")

# On-demand stream — no shard management
kinesis.create_stream(StreamName="clickstream",
                      StreamModeDetails={"StreamMode": "ON_DEMAND"})

# Batched put — partition key picks the shard; high-cardinality keys avoid hot shards
kinesis.put_records(
    StreamName="clickstream",
    Records=[
        {"Data": json.dumps({"userId": "u-001", "page": "/checkout"}).encode(), "PartitionKey": "u-001"},
        {"Data": json.dumps({"userId": "u-002", "page": "/home"}).encode(),     "PartitionKey": "u-002"},
    ],
)

# Firehose stream — Kinesis source, S3 sink, partitioned by event time, Parquet-converted
firehose.create_delivery_stream(
    DeliveryStreamName="clickstream-to-s3",
    DeliveryStreamType="KinesisStreamAsSource",
    KinesisStreamSourceConfiguration={
        "KinesisStreamARN": "arn:aws:kinesis:us-east-1:111122223333:stream/clickstream",
        "RoleARN":          "arn:aws:iam::111122223333:role/FirehoseRole",
    },
    ExtendedS3DestinationConfiguration={
        "RoleARN":    "arn:aws:iam::111122223333:role/FirehoseRole",
        "BucketARN":  "arn:aws:s3:::my-data-lake",
        "Prefix":     "clickstream/year=!{timestamp:yyyy}/month=!{timestamp:MM}/day=!{timestamp:dd}/",
        "BufferingHints":     {"SizeInMBs": 64, "IntervalInSeconds": 300},
        "CompressionFormat":  "GZIP",
    },
)

# Act 3 — Orchestrating: Step Functions

Decoupling lets components pass messages independently. But sometimes the work is not a single message that triggers a single handler — it is a multi-step process. Validate the order, check inventory, branch on the result, charge the card, notify the customer, reserve the inventory. Each step might fail and need its own retry. Some steps need to run in parallel. One step might wait days for a human to click an approval link.

Stitching that together as chained Lambda calls turns into a mess by month three — inconsistent retries, no visibility into which step failed, and a fifteen-minute ceiling. **Step Functions** is the managed answer — a state machine that sequences tasks, branches, retries, runs in parallel, and waits for external callbacks, with the full execution history visible in the console.

## Step Functions — Workflow Orchestration

A workflow that chains a few Lambda functions usually starts as Lambda A calling Lambda B calling Lambda C. It works on day one and turns into a swamp by month three: every function has its own retry logic, error handling is inconsistent, there's no way to see which step failed, and long-running flows hit Lambda's 15-minute ceiling.

Step Functions is the managed answer. You describe the workflow as a **state machine** in **Amazon States Language (ASL)** — a JSON definition of states and transitions. Step Functions runs the state machine, invokes each step, retries on failure with configured policies, and shows the full execution history in the console.

### State types

| State | Does |
|---|---|
| **Task** | invoke a Lambda or call an AWS service directly |
| **Choice** | branch on input data (if/else) |
| **Wait** | pause for a duration or until a timestamp |
| **Parallel** | run multiple branches simultaneously; wait for all |
| **Map** | iterate over an array, apply a sub-workflow to each item |
| **Pass** | pass input to output unchanged (handy for injecting constants) |
| **Succeed** / **Fail** | end execution |

The Task state is the workhorse. It can invoke a Lambda, but it can also call **over 200 AWS services directly** through service integrations — `dynamodb:PutItem`, `sqs:SendMessage`, `sns:Publish`, `ecs:RunTask`, `glue:StartJobRun`, `bedrock:InvokeModel`. The direct integrations remove the need for Lambda functions whose only job is to glue services together — write the AWS call into the workflow itself.

Three integration patterns matter for Task states:

- **Request-Response** (default) — call the service, move to the next state immediately
- **Run a Job (`.sync`)** — call the service, **wait for the job to finish** (ECS task completion, Glue job done, EMR step finished) before moving on
- **Wait for Task Token (`.waitForTaskToken`)** — pause the workflow, hand out a callback token, resume when something external (a human approval, a third-party API) calls `SendTaskSuccess`

## Retry, Catch, and Failure Modes

The error-handling story is the main reason Step Functions exists. Two primitives.

**Retry.** A Task state can attach a list of retry rules. Each rule names a set of error types (`Lambda.ServiceException`, `States.Timeout`, `States.TaskFailed`, or `States.ALL` for everything), an initial interval, a max attempt count, and a backoff rate. A typical config — 2 seconds, 3 attempts, backoff 2× — retries at 2 s, 4 s, 8 s before giving up. Most transient failures (Lambda throttles, brief network blips, DynamoDB ProvisionedThroughputExceeded) are absorbed by retries with no application code involved.

**Catch.** When retries are exhausted, a Catch clause routes to a specific error-handling state — "on any failure, run `NotifyFailure` and end". The `ResultPath` controls where the error detail is injected into the state input, so the handler can read it. Together, Retry and Catch turn what would be a try/catch tangle in application code into a declarative part of the workflow definition.

### Wait for Task Token

The most useful integration pattern, and the one people miss most often. A Task state with `.waitForTaskToken` sends a task token to a downstream system (an SQS message, a Lambda invocation, an HTTP call) and **pauses the workflow** until that system calls `SendTaskSuccess(taskToken, output)` (or `SendTaskFailure`). The workflow then resumes with the returned output.

This is the right shape for human-in-the-loop approvals (a manager clicks an approval link, which calls `SendTaskSuccess`), long-running third-party processes (a payment processor finishes hours later and calls back), or any "wait for an external thing to happen" pattern. Step Functions can hold the wait for **up to a year** on a Standard workflow, with no infrastructure cost during the wait.

## Standard vs Express Workflows

Two workflow types, very different cost and execution models.

| | Standard | Express |
|---|---|---|
| Max duration | **1 year** | 5 minutes |
| Execution model | **exactly-once** per state | at-least-once (async) or at-most-once (sync) |
| Pricing | per state transition | per execution duration + requests |
| History | stored in Step Functions, viewable forever | logged to CloudWatch Logs |
| Throughput | up to ~2,000 starts/sec | 100,000+ starts/sec |

**Standard** is the right pick for business workflows — order fulfilment, data pipelines, human approvals, anything where each step has to run exactly once, the audit trail matters, and the workflow might be open for hours, days, or longer. Pricing is per state transition, which is cheap on long-running flows that don't transition often.

**Express** is the right pick for high-throughput, short-duration event handlers — IoT ingestion, per-request microservice orchestration, fast data transforms. Up to 5 minutes per execution, but at 100,000+ executions per second. Pricing is by duration and request count, which is cheap on short executions but punishing if you accidentally use it for long workflows.

The rule: long, careful, auditable → Standard. Short, frequent, high-volume → Express. You can also call an Express workflow as a step inside a Standard workflow when you want a high-throughput sub-step inside an audited parent.

# Act 4 — Routing: EventBridge

SNS does pub/sub by subscription — one topic, many subscribers, optionally filtered. Step Functions orchestrates multi-step workflows. What if your problem is the routing layer itself? You have many event sources — AWS services, Sass partners, your own application — and many targets, and you need rules that decide which targets see which events based on the event content.

That is an event bus, and **EventBridge** is what AWS provides. Where SNS does pub/sub by subscription, EventBridge does routing by pattern match — the rule is the abstraction, not the subscription. This act covers the bus model, schema registry, Pipes for source-to-target integrations without glue Lambda, archive and replay, and how EventBridge ties together with Step Functions in a real architecture.

## EventBridge — the Event Bus

EventBridge is a serverless **event bus** that routes events to targets based on **rules**. Where SNS does pub/sub by subscription, EventBridge does routing by pattern match — the rule is the abstraction, not the subscription.

### Three kinds of bus

- **Default event bus** — every account has one; AWS services publish service events here automatically (EC2 state changes, S3 object events, CodePipeline transitions, Health events, IAM activity, hundreds more)
- **Custom event bus** — for your own application's events, populated via `PutEvents`
- **Partner event bus** — SaaS partners (Datadog, Zendesk, GitHub, Stripe, PagerDuty, Auth0, MongoDB Atlas, many more) publish their events into a partner bus in your account

An event is a JSON object with a small set of standard fields (`source`, `detail-type`, `account`, `region`, `time`) and a free-form `detail` payload — the same shape across AWS-service, SaaS, and custom events.

### Rules — pattern matching

A **rule** has an event pattern that filters incoming events and a list of up to **five targets** that matching events are delivered to. Patterns match on any field with a rich syntax — exact values, lists of allowed values, prefix matching, numeric ranges, anything-but, existence checks, and wildcards. Patterns are JSON, evaluated cheaply at the bus layer before any target is invoked.

Targets cover most of the AWS surface — Lambda, Step Functions, SQS, SNS, Kinesis, Firehose, ECS run-task, CodePipeline, CodeBuild, EventBridge buses in other accounts or regions, **API destinations** (any HTTP endpoint with auth), even SSM parameter updates. Per-target **input transformers** reshape the event before delivery — pull specific fields out, restructure the JSON, inject constants — which removes a lot of "Lambda just to massage the event shape" glue functions.

### Scheduled rules

A rule can also fire on a **schedule** instead of matching an event — a `cron` or `rate` expression. `rate(5 minutes)`, `cron(0 8 ? * MON-FRI *)`. The standard replacement for cron jobs on a box: every EventBridge schedule can call Lambda, Step Functions, ECS, or any other target. **EventBridge Scheduler** is the newer dedicated scheduling product, with higher scale and one-time schedules — preferred over scheduled rules for any new scheduling work, but scheduled rules remain everywhere.

EventBridge **replaces CloudWatch Events** — same default bus, same rules, same patterns. CloudWatch Events is the older name; EventBridge is what it became when AWS extended it with custom buses, SaaS partners, and the broader integration surface.

## EventBridge — Schema Registry, Pipes, Archive

Three features that show up in real architectures.

**Schema Registry.** EventBridge auto-discovers the schema of events flowing through a bus and stores them in a registry. From there you can generate **code bindings** — Python, Java, TypeScript — so application code consumes events as typed objects instead of raw JSON dictionaries. Useful as the event surface grows past a handful of types and stops fitting in one engineer's head.

**EventBridge Pipes.** A point-to-point integration between a source and a target with optional filtering and enrichment, no glue code. Pattern: SQS / Kinesis / DynamoDB Streams / MSK / self-managed Kafka / MQ → optional filter → optional enrichment Lambda → target (Step Functions, API Gateway, SQS, SNS, EventBridge bus, Lambda, anywhere). Pipes replace the very common "Lambda function that polls one service and forwards to another" pattern. Cheaper, less to operate, and the filter step runs at the source so unwanted records don't hit the enrichment Lambda at all.

**Archive and Replay.** EventBridge can archive every event matching a pattern for a configured retention period. Replay sends archived events back to a bus, with optional filtering. The use cases are debugging ("replay last Tuesday's order events through the new fraud detector"), onboarding new consumers ("feed the last 30 days through the new system before going live"), and incident recovery ("the bug that swallowed 4 hours of events is fixed — replay").

### Cross-account and cross-region

A bus in one account can deliver events to a bus in another account, and to buses in other regions. The standard pattern is a **central event bus** in a logging or security account that receives events from every account in the organisation — one place to attach detective rules, one place to ship to SIEM, one place to react to incidents.

In [ ]:
import boto3, json

sfn = boto3.client("stepfunctions", region_name="us-east-1")
eb  = boto3.client("events",         region_name="us-east-1")

# Order workflow: validate, branch on inventory, charge, then notify and reserve in parallel
workflow = {
    "StartAt": "ValidateOrder",
    "States": {
        "ValidateOrder": {
            "Type": "Task",
            "Resource": "arn:aws:lambda:us-east-1:111122223333:function:validate-order",
            "Retry":  [{"ErrorEquals": ["Lambda.ServiceException"], "IntervalSeconds": 2,
                        "MaxAttempts": 3, "BackoffRate": 2}],
            "Catch":  [{"ErrorEquals": ["States.ALL"], "Next": "OrderFailed",
                        "ResultPath": "$.error"}],
            "Next": "CheckInventory",
        },
        "CheckInventory": {
            "Type": "Choice",
            "Choices": [{"Variable": "$.inventoryAvailable", "BooleanEquals": True,
                         "Next": "ProcessPayment"}],
            "Default": "OutOfStock",
        },
        "ProcessPayment": {
            "Type": "Task",
            "Resource": "arn:aws:lambda:us-east-1:111122223333:function:process-payment",
            "Next": "NotifyAndReserve",
        },
        "NotifyAndReserve": {
            "Type": "Parallel", "End": True,
            "Branches": [
                {"StartAt": "Notify", "States": {"Notify": {
                    "Type": "Task", "Resource": "arn:aws:states:::sns:publish",
                    "Parameters": {"TopicArn": "arn:aws:sns:us-east-1:111122223333:orders",
                                   "Message.$": "$.orderId"}, "End": True}}},
                {"StartAt": "Reserve", "States": {"Reserve": {
                    "Type": "Task", "Resource": "arn:aws:states:::dynamodb:updateItem",
                    "Parameters": {"TableName": "Inventory",
                                   "Key": {"itemId": {"S.$": "$.itemId"}},
                                   "UpdateExpression": "SET reserved = reserved + :one",
                                   "ExpressionAttributeValues": {":one": {"N": "1"}}},
                    "End": True}}},
            ],
        },
        "OutOfStock":  {"Type": "Fail", "Error": "OutOfStock"},
        "OrderFailed": {"Type": "Fail", "Error": "ValidationFailed"},
    },
}

sm = sfn.create_state_machine(
    name="OrderFulfilment", definition=json.dumps(workflow),
    roleArn="arn:aws:iam::111122223333:role/StepFunctionsRole", type="STANDARD",
)

# EventBridge rule — start the workflow when a custom OrderPlaced event arrives
eb.put_rule(
    Name="OnOrderPlaced",
    EventPattern=json.dumps({
        "source":      ["myapp.orders"],
        "detail-type": ["OrderPlaced"],
        "detail": {"status": ["new"], "amount": [{"numeric": [">", 0]}]},
    }),
    State="ENABLED",
)
eb.put_targets(Rule="OnOrderPlaced", Targets=[{
    "Id": "StartWorkflow", "Arn": sm["stateMachineArn"],
    "RoleArn": "arn:aws:iam::111122223333:role/EventBridgeStepFunctionsRole",
    "InputTransformer": {
        "InputPathsMap": {"orderId": "$.detail.orderId", "itemId": "$.detail.itemId",
                          "amount":  "$.detail.amount"},
        "InputTemplate": '{"orderId": "<orderId>", "itemId": "<itemId>", "amount": <amount>}',
    },
}])

# Act 5 — Picking the right tool

Five mechanisms on the table, all answering different versions of the same question — how do components pass work to each other without coupling. The shape behind each one is different, and the skill is matching the shape to the problem.

## Picking the Right Tool

The whole notebook in one decision sheet.

| Need | Service |
|---|---|
| Decouple producer/consumer, buffer work, one consumer per message | **SQS Standard** |
| Same, with strict order and exactly-once | **SQS FIFO** |
| Isolate failing messages without blocking the queue | **SQS DLQ** |
| One event → many subscribers at once | **SNS** |
| Same, with ordering | **SNS FIFO** (subscribers must be SQS FIFO) |
| Durable fan-out of one event to many independent workers | **SNS + SQS** |
| Real-time stream, many consumers, replay history | **Kinesis Data Streams** |
| Bulk-deliver streaming data to S3 / Redshift / OpenSearch | **Kinesis Data Firehose** |
| SQL or Flink over streams (windows, joins, anomalies) | **Managed Service for Apache Flink** |
| Streaming on Apache Kafka semantics or existing Kafka clients | **MSK** (or MSK Serverless) |
| Multi-step workflow with branches, retries, parallel, audit trail | **Step Functions Standard** |
| Very high-throughput short workflow (≤ 5 min) | **Step Functions Express** |
| Pause workflow for human approval or external callback | **Step Functions `.waitForTaskToken`** |
| Route AWS / SaaS / custom events to many targets by rule | **EventBridge** |
| Scheduled jobs (cron / rate) | **EventBridge Scheduler** (or scheduled rule) |
| Source → filter → enrich → target with no glue Lambda | **EventBridge Pipes** |

The shape behind all of these is the same — get producers and consumers off each other's hot path, let each side scale and fail independently, get observability and retries from the platform. The skill is recognising which mechanism the workload actually needs: a queue, a fan-out, a stream, a workflow, or a routed bus.